# 11 · PagedAttention and the block allocator
Level L3 · Part 2 · core · ~25 min · builds on 10

## Goal
By the end of this lesson you can explain block tables and fragmentation, read the real block pool that vLLM uses from its pinned image, and build the same allocator by hand in about 30 lines, so that you know what it does at every step.

## Why it matters
- **Without it:** the engine's KV memory is a black box. When the log says "no free blocks", you cannot tell whether the pool is too small, the blocks are too big, or the answers are too long.
- **With it:** the pool is a free list of block ids plus a block table per request. A request takes a new block only when its last block is full, and finishing returns all its blocks at once. The waste per request is at most one partly filled block (15 empty slots at block size 16). When the free list is empty, the failure is visible in one place, and the engine has to preempt a request (lesson 13).
- **When you don't need it:** a toy engine that serves one request at a time, or a job where every answer has the same known length. A fixed reservation is fine there.

## The idea
Think of a warehouse that stores each order in fixed-size boxes, not in a room sized for the largest order it could ever get. A box is handed out when the last one is full. A card per order (the block table) lists which boxes hold its items, in order. When the order ships, its boxes go back on the shelf. vLLM's PagedAttention does this with the KV cache: a block is a fixed group of token slots (16 by default), each request keeps a block table that maps its token positions to physical blocks, and a free list holds the blocks nobody uses. The name comes from operating-system paging: the attention kernel reads a request's KV through its table, so the blocks do not have to sit next to each other in memory.

## Picture
```mermaid
flowchart LR
  F["free list: blocks nobody uses"] -->|"take one when the last block is full"| T
  T["block table of request 1: blocks 7, 6, 5"] --> B7["block 7: tokens 0 to 15"]
  T --> B6["block 6: tokens 16 to 31"]
  T --> B5["block 5: tokens 32 to 39, 8 of 16 used"]
  B5 -->|"request finishes: all blocks return"| F
```

## Step by step
### Step 1: vLLM's own block pool (the library version)
The pinned vLLM image (`vllm/vllm-openai-cpu:v0.32.0`, the CPU build) contains the real block pool. This cell starts the image's Python, imports `CacheConfig` and `BlockPool`, and prints what the engine uses. No GPU and no model download are needed. The first run pulls the image, which takes a few minutes; later runs take seconds. If Docker is not enabled on this machine, the next cell prints the recorded output from the same run.

In [1]:
LIBRARY_PROBE = """
from vllm.config import CacheConfig
from vllm.v1.core.block_pool import BlockPool
print('default block_size:', CacheConfig().block_size)
pool = BlockPool(num_gpu_blocks=10, enable_caching=False, hash_block_size=16)
print('pool of 10 blocks, free at start:', pool.get_num_free_blocks())
blocks = pool.get_new_blocks(3)
print('handed out ids:', [b.block_id for b in blocks], '| free now:', pool.get_num_free_blocks())
pool.free_blocks(blocks)
print('after free_blocks:', pool.get_num_free_blocks(), '| usage:', pool.get_usage())
"""

**Run the probe in the pinned image.** Docker starts the image with its Python, and `-I` keeps the run isolated from the working folder. This cell is skipped unless Docker is enabled.

In [2]:
import subprocess
from pathlib import Path

IMAGE = "vllm/vllm-openai-cpu:v0.32.0"           # the pinned CPU image (lesson 08 uses the same tag)
Path("../data/11").mkdir(parents=True, exist_ok=True)
run = subprocess.run(["docker", "run", "--rm", "--entrypoint", "python3", IMAGE, "-I", "-c", LIBRARY_PROBE],
                     capture_output=True, text=True, timeout=900)
if run.returncode == 0:
    Path("../data/11/library_probe.txt").write_text(run.stdout)
    print(run.stdout)
else:
    print("docker run failed:", run.stderr[-300:])

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**The same output without Docker.** The next cell reads the live output if the cell above ran, and otherwise the recorded copy in `data/recorded/11.json`. The lesson reads the same on a laptop with no Docker.

In [3]:
import json
from pathlib import Path
live = Path("../data/11/library_probe.txt")
if live.exists():
    print("live output from the pinned image:")
    print(live.read_text())
else:
    recorded = json.loads(Path("../data/recorded/11.json").read_text())
    print("recorded output (from a real run of the same image, " + recorded["image"] + "):")
    print(recorded["stdout"])

live output from the pinned image:
default block_size: 16
pool of 10 blocks, free at start: 9
handed out ids: [1, 2, 3] | free now: 6
after free_blocks: 9 | usage: 0.0



**What you should see:** `default block_size: 16`. One block is kept back, so a pool of 10 gives 9 usable blocks, and the ids handed out start at 1. Freeing blocks puts the count back. This is the real engine's behaviour; the next section builds the same thing by hand.

## Under the hood
Step 1 used vLLM's pool. Here is the mechanism in one class: a free list, a block table per request, and one rule for when a request takes a block. It has no prefix sharing, no reference counts and no eviction. Those come in lessons 13 and 14. The class is about 30 lines.

In [4]:
class OutOfBlocks(Exception):
    """The pool has no free block. The engine must preempt a request (lesson 13)."""

class BlockPool:
    def __init__(self, num_blocks, block_size=16):
        self.block_size = block_size
        self.free = list(range(num_blocks))    # free list: ids of blocks nobody uses
        self.tables = {}                       # request -> block table (block ids, in token order)
        self.length = {}                       # request -> tokens stored so far

    def append_token(self, req):
        n = self.length.get(req, 0)
        if n % self.block_size == 0:           # no block yet, or the last one is full: take one
            if not self.free:
                raise OutOfBlocks(req)
            self.tables.setdefault(req, []).append(self.free.pop())
        self.length[req] = n + 1

    def slot_of(self, req, pos):               # where token number pos lives: (block id, offset)
        return self.tables[req][pos // self.block_size], pos % self.block_size

    def finish(self, req):                     # the request is done: all its blocks go back at once
        self.free.extend(self.tables.pop(req, []))
        self.length.pop(req, None)

    def waste(self):                           # reserved slots that hold no token (internal fragmentation)
        return sum(len(t) * self.block_size - self.length[r] for r, t in self.tables.items())

### One request, token by token
A request with 40 tokens needs three blocks: 16 + 16 + 8. Watch its block table grow, find one token's slot, then finish it.

In [5]:
pool = BlockPool(num_blocks=8)
for _ in range(40):
    pool.append_token("req-1")
print("block table:", pool.tables["req-1"], "| tokens:", pool.length["req-1"])
print("token 35 lives in (block, offset):", pool.slot_of("req-1", 35))
print(f"free blocks: {len(pool.free)} | empty slots: {pool.waste()}")
pool.finish("req-1")
print("after finish, free blocks:", len(pool.free))

block table: [7, 6, 5] | tokens: 40
token 35 lives in (block, offset): (5, 3)
free blocks: 5 | empty slots: 8
after finish, free blocks: 8


**What you should see:** the table is `[7, 6, 5]`: the pool hands out the highest free id first. Token 35 sits at block 5, offset 3. Eight slots are empty, all in the last block. Finishing returns the three blocks, so all eight are free again.

### The trace from lesson 10, on a real budget
Lesson 10 counted blocks for the support-desk trace. The cells below restate its numbers (the budget from lesson 05, the same made-up trace) and push the requests through the pool one token at a time.

In [6]:
import random

GIB = 2**30
KV_PER_TOKEN = 131_072                  # Llama 3.1 8B, FP16: 128 KiB per token (lesson 03)
GPU = 24e9 * 0.92                       # gpu_memory_utilization 0.92 on one 24 GB L4
WEIGHTS = 8_030_261_248 * 2             # 8.03 billion parameters x 2 bytes
BUDGET_TOKENS = int((GPU - WEIGHTS - 1 * GIB) // KV_PER_TOKEN)   # about 37,700 tokens

def make_trace(n=200, seed=7, policy=1_500, max_len=4_096):
    """The made-up support-desk trace from lesson 10: same seed, same shape."""
    rng = random.Random(seed)
    rows = []
    for i in range(n):
        question = rng.randint(30, 200)
        answer = rng.randint(50, 250) if rng.random() < 0.8 else rng.randint(400, 1_200)
        rows.append({"id": i, "total": min(policy + question + answer, max_len)})
    return rows

trace = make_trace()
print(f"{len(trace)} requests in the trace; budget {BUDGET_TOKENS:,} tokens = {BUDGET_TOKENS // 16:,} blocks")

200 requests in the trace; budget 37,732 tokens = 2,358 blocks


Now admit the requests in order, one token at a time, until the pool runs out. A request that runs out is dropped and its blocks are returned.

In [7]:
def admit_by_hand(num_blocks, rows, block_size=16):
    """Admit requests in order, one token at a time, until the pool runs out."""
    pool, admitted = BlockPool(num_blocks, block_size), []
    for row in rows:
        try:
            for _ in range(row["total"]):
                pool.append_token(row["id"])
            admitted.append(row)
        except OutOfBlocks:
            held = len(pool.tables.get(row["id"], []))
            pool.finish(row["id"])          # the request that did not fit gives its blocks back
            print(f"request {row['id']} ran out of blocks after taking {held}; it was dropped")
            break
    return pool, admitted

pool, admitted = admit_by_hand(BUDGET_TOKENS // 16, trace)
live_tokens = sum(pool.length[r["id"]] for r in admitted)
live_slots = sum(len(pool.tables[r["id"]]) * 16 for r in admitted)
print(f"admitted {len(admitted)} requests, free blocks {len(pool.free)} of {BUDGET_TOKENS // 16}")
print(f"empty slots in live blocks: {pool.waste()} of {live_slots} ({pool.waste() / live_slots:.1%})")

request 21 ran out of blocks after taking 5; it was dropped
admitted 21 requests, free blocks 5 of 2358
empty slots in live blocks: 166 of 37648 (0.4%)


**What you should see:** 21 requests fit, the same count lesson 10 found. The 22nd takes the last 5 free blocks, runs out in the middle of its answer, and gives them back. Empty slots are a fraction of a percent, because the only waste is the tail of each request's last block. Lesson 10's reservation left more than half of its slots empty.

## At work
- **Cloudflare** reports that an `x-session-affinity` header lifted the peak input cache hit ratio from 60% to 80%. Cached prefixes live in blocks on one engine, so routing a session back to the same engine keeps its blocks warm. → lesson 14 (prefix caching) and lesson 30 (routing).
- **vLLM** exports `vllm:kv_cache_usage_perc`. A value near 1.0 means the free list is almost empty, and the next long answer will trigger preemption. Your dashboard needs this number next to the queue depth. → lesson 33.
- **Preemption storms** are the usual failure when the pool stays full. The scheduler recomputes requests that lost their blocks, which spends prefill compute and raises tail latency. The first sign is a rising preemption count while the load stays flat. → lesson 13.

Sources: vLLM design docs and metrics page (docs.vllm.ai); Cloudflare, "High-performance LLMs" (2026-05): https://blog.cloudflare.com/high-performance-llms/

## Common mistakes
**1. Reading a token past the end of its block table.** A request with 40 tokens has three blocks. Asking for token 60 points at a fourth block that the request does not own. Check the length first.

In [8]:
pool = BlockPool(num_blocks=8)
for _ in range(40):
    pool.append_token("req-2")
pool.slot_of("req-2", 60)               # IndexError: the table has three blocks, position 60 needs block index 3

IndexError: list index out of range

**2. Forgetting `finish()`.** A request that ends but keeps its blocks leaks them. The pool fills with dead requests, and the next live request gets `OutOfBlocks`. The fix is to free the blocks on every exit path, including errors and client disconnects.

In [9]:
leaky = BlockPool(num_blocks=4)
for req in ["a", "b", "c", "d"]:          # none of them calls finish()
    leaky.append_token(req)
for _ in range(16):
    leaky.append_token("e")               # OutOfBlocks: all four blocks are held by requests that are gone

OutOfBlocks: e

**3. Thinking a block is 16 bytes or 16 requests.** A block is 16 token slots. Its size in bytes is 16 times the per-token KV size: 16 x 128 KiB = 2 MiB for Llama 3.1 8B in FP16. Block size changes the granularity of waste and the number of blocks, not how much each token costs.

## Try it
Run the same trace with a block size of 32 instead of 16. Predict first: will more or fewer requests fit on the same budget, and will the empty slots go up or down? Then run it and compare with the 16-token result above.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
for size in (16, 32):
    pool, admitted = admit_by_hand(BUDGET_TOKENS // size, trace, block_size=size)
    slots = sum(len(pool.tables[r["id"]]) * size for r in admitted)
    print(f"block {size:2}: {len(admitted)} requests, empty slots {pool.waste() / slots:.1%}")

request 21 ran out of blocks after taking 5; it was dropped
block 16: 21 requests, empty slots 0.4%
request 20 ran out of blocks after taking 55; it was dropped
block 32: 20 requests, empty slots 0.8%


## Recap
- A block is a fixed group of token slots (16 by default in vLLM). A request keeps a block table that maps token positions to block ids.
- A request takes a block when its last one is full, and gives them all back on finish.
- Waste per request is at most one partly filled block, so the same budget admits about twice as many requests as a reservation (21 against 9).
- When the pool runs out mid-answer, the engine preempts (lesson 13). Prefix reuse (lesson 14) builds on the same blocks.

**Interview:** "How can a vLLM-style engine keep a request's KV cache in non-contiguous memory, and what is the waste bounded by?" Strong answer: an ordered list of fixed-size block ids; token i sits at `table[i // B]`, offset `i % B`; waste is at most B minus 1 slots per request; when the free list is empty, the scheduler preempts.